# §13.7.6 — 길이에 따른 메모리와 시간의 실측

> 딥러닝 교재 · 3부 13장 7절 6항 (🐍)
> 선행: §13.7.1(복잡도 계산) · §13.7.2(선형 어텐션) · §13.7.3·§13.7.5(온라인 소프트맥스)

## 이 노트북이 답하는 질문

1. **추가 메모리는 정말 $O(T^2)$ 대 $O(T)$로 갈리는가?**
2. **온라인 소프트맥스는 정확하고 선형 어텐션은 근사인가?** 출력 오차로 판정한다.
3. **복잡도 개선은 벽시계 시간으로 이어지는가?** 그리고 근사의 대가는 어디서 드러나는가?

**예상 실행 시간** CPU 약 60초.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 세 구현 — 같은 입력, 다른 계산

* **표준**: $T\times T$ 로짓·어텐션 행렬을 통째로 만든다.
* **스트리밍(온라인 소프트맥스)**: §13.7.5의 갱신식 그대로, 키·값을 블록으로 흘리며
  $T\times T$ 행렬 없이 계산한다. **같은 함수의 다른 평가 순서**다.
* **선형(ELU+1 커널)**: $\exp(q^\top k)$를 $\phi(q)^\top\phi(k)$로 대체하고 결합법칙으로
  순서를 바꾼다(정리 13.7.2). **다른 함수**다.

메모리는 만들어지는 중간 배열의 바이트를 직접 합산해 잰다(입출력 $Q,K,V,Y$ 제외).

In [ ]:
def std_attn(Q, K, V, scale):
    T_, d = Q.shape
    E = Q @ K.T / scale                     # (T,T)
    E -= E.max(1, keepdims=True)
    A = np.exp(E); A /= A.sum(1, keepdims=True)
    Y = A @ V
    mem = E.nbytes + A.nbytes
    return Y, mem

def stream_attn(Q, K, V, scale, block=64):
    T_, d = Q.shape
    m = np.full(T_, -np.inf); l = np.zeros(T_); o = np.zeros((T_, V.shape[1]))
    peak = 0
    for s0 in range(0, T_, block):
        Kb, Vb = K[s0:s0 + block], V[s0:s0 + block]
        Eb = Q @ Kb.T / scale               # (T, block)
        mb = Eb.max(1)
        pb = np.exp(Eb - mb[:, None])
        lb = pb.sum(1)
        ob = pb @ Vb
        mn = np.maximum(m, mb)
        c, cb = np.exp(m - mn), np.exp(mb - mn)
        l = c * l + cb * lb
        o = c[:, None] * o + cb[:, None] * ob
        m = mn
        peak = max(peak, Eb.nbytes + pb.nbytes + ob.nbytes)
    mem = peak + m.nbytes + l.nbytes + o.nbytes
    return o / l[:, None], mem

def phi(x):
    return np.where(x > 0, x + 1.0, np.exp(np.minimum(x, 0)))

def linear_attn(Q, K, V, scale):
    Qf, Kf = phi(Q / np.sqrt(scale)), phi(K / np.sqrt(scale))
    S_ = Kf.T @ V                            # (d, d_v)
    z = Kf.sum(0)                            # (d,)
    Y = (Qf @ S_) / (Qf @ z)[:, None]
    mem = Qf.nbytes + Kf.nbytes + S_.nbytes + z.nbytes
    return Y, mem

d = 64
rn = np.random.default_rng(SEED)
Qs, Ks, Vs = (rn.standard_normal((256, d)) for _ in range(3))
Y0, _ = std_attn(Qs, Ks, Vs, np.sqrt(d))
Y1, _ = stream_attn(Qs, Ks, Vs, np.sqrt(d))
Y2, _ = linear_attn(Qs, Ks, Vs, np.sqrt(d))
print(f"스트리밍 오차 |Y-Y_std|_max = {np.abs(Y1-Y0).max():.2e}   ← 머신 엡실론 수준")
print(f"선형   오차 |Y-Y_std|_max = {np.abs(Y2-Y0).max():.2e}   ← 다른 함수")

---
## 2. 길이를 훑는다 — 메모리·오차·시간

In [ ]:
T_GRID = [64, 256, 1024] if FAST else [64, 128, 256, 512, 1024, 2048]
mems = {k: [] for k in ['std', 'stream', 'lin']}
errs = {k: [] for k in ['stream', 'lin']}
times = {k: [] for k in ['std', 'stream', 'lin']}
for T in T_GRID:
    Q, K, V = (np.random.default_rng(T).standard_normal((T, d)) for _ in range(3))
    for name, fn in [('std', std_attn), ('stream', stream_attn), ('lin', linear_attn)]:
        ts = []
        for _ in range(3):
            t0 = time.perf_counter()
            Y, mem = fn(Q, K, V, np.sqrt(d))
            ts.append(time.perf_counter() - t0)
        times[name].append(min(ts)); mems[name].append(mem)
        if name == 'std':
            Yref = Y
        else:
            errs[name].append(np.abs(Y - Yref).max())
    print(f"T={T:5d}:  메모리 std {mems['std'][-1]/2**20:6.1f}MiB | stream {mems['stream'][-1]/2**20:6.2f}MiB"
          f"  시간 std {times['std'][-1]*1e3:6.1f}ms | stream {times['stream'][-1]*1e3:6.1f}ms | lin {times['lin'][-1]*1e3:5.1f}ms")

---
## 3. 근사의 대가 — 정밀 회상

키를 무작위 단위벡터로 두고 질의를 그중 하나($k_{j^*}$의 $\beta$배)로 주면, 표준
어텐션은 소프트맥스의 지수적 분리로 $v_{j^*}$를 거의 정확히 인출한다. 선형 커널은
그 지수적 분리가 없어, 후보가 늘수록 나머지 항목들의 잔여 가중치가 인출을 오염시킨다.

In [ ]:
BETA = 20.0
T_REC = T_GRID
hit = {'std': [], 'lin': []}
verr = {'std': [], 'lin': []}
for T in T_REC:
    rn2 = np.random.default_rng(T + 1)
    K2 = rn2.standard_normal((T, d)); K2 /= np.linalg.norm(K2, axis=1, keepdims=True)
    V2 = rn2.standard_normal((T, d))
    js = rn2.integers(0, T, 128)
    Q2 = BETA * K2[js]
    for name, fn in [('std', std_attn), ('lin', linear_attn)]:
        Y, _ = fn(Q2, K2, V2, 1.0)          # 이미 정규화된 키 — 스케일 1
        vt = V2[js]
        verr[name].append(np.linalg.norm(Y - vt, axis=1).mean() / np.linalg.norm(vt, axis=1).mean())
    print(f"T={T:5d}: 상대 인출 오차  std {verr['std'][-1]:.3f} | lin {verr['lin'][-1]:.3f}")

---
## 4. 교재 그림 — fig_13_7_6

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 추가 메모리
ax = axes[0]
ax.loglog(T_GRID, np.array(mems['std'])/2**20, 'o-', color=CB[4], ms=5, label=lab('표준 ($T\\times T$ 보관)', 'standard'))
ax.loglog(T_GRID, np.array(mems['stream'])/2**20, 's-', color=CB[5], ms=5, label=lab('스트리밍 (블록+통계량)', 'streaming'))
ax.loglog(T_GRID, np.array(mems['lin'])/2**20, '^-', color=CB[3], ms=5, label=lab('선형 (상태 $S,z$)', 'linear'))
ref = np.array(T_GRID, float)
ax.loglog(ref, (ref/ref[0])**2 * mems['std'][0]/2**20, ':', color='0.6', lw=0.8)
ax.text(ref[-2], (ref[-2]/ref[0])**2 * mems['std'][0]/2**20*2.0, '$\\propto T^2$', fontsize=8, color='0.4')
ax.set_xlabel(lab('길이 $T$', 'length')); ax.set_ylabel(lab('추가 메모리 (MiB)', 'extra memory'))
ax.set_title(lab('(a) $O(T^2)$ 보관 대 $O(T)$ 스트리밍', '(a) memory'), fontsize=10)
ax.legend(fontsize=8)

# (b) 표준 대비 출력 오차
ax = axes[1]
ax.loglog(T_GRID, errs['stream'], 's-', color=CB[5], ms=5, label=lab('온라인 소프트맥스', 'online softmax'))
ax.loglog(T_GRID, errs['lin'], '^-', color=CB[3], ms=5, label=lab('선형 어텐션', 'linear attn'))
ax.axhline(np.finfo(float).eps, color='k', lw=0.7, ls=':')
ax.text(T_GRID[0], np.finfo(float).eps*2.2, lab('머신 엡실론', 'machine eps'), fontsize=8)
ax.set_xlabel(lab('길이 $T$', 'length')); ax.set_ylabel(lab('$\\|Y-Y_{\\rm std}\\|_\\infty$', 'error'))
ax.set_title(lab('(b) 하나는 같은 함수, 하나는 다른 함수', '(b) exactness'), fontsize=10)
ax.legend(fontsize=8, loc='center right')

# (c) 벽시계 시간
ax = axes[2]
for name, col, mk, lb in [('std', CB[4], 'o', lab('표준', 'standard')),
                          ('stream', CB[5], 's', lab('스트리밍', 'streaming')),
                          ('lin', CB[3], '^', lab('선형', 'linear'))]:
    ax.loglog(T_GRID, np.array(times[name])*1e3, mk+'-', color=col, ms=5, label=lb)
ax.set_xlabel(lab('길이 $T$', 'length')); ax.set_ylabel(lab('시간 (ms)', 'time (ms)'))
ax.set_title(lab('(c) 복잡도 개선 ≠ 벽시계 개선 (단일 스레드)', '(c) wall clock'), fontsize=10)
ax.legend(fontsize=8)

# (d) 정밀 회상
ax = axes[3]
ax.semilogx(T_REC, verr['std'], 'o-', color=CB[4], ms=5, label=lab('표준 (소프트맥스)', 'softmax'))
ax.semilogx(T_REC, verr['lin'], '^-', color=CB[3], ms=5, label=lab('선형 (ELU+1 커널)', 'linear'))
ax.set_xlabel(lab('후보 수 $T$', 'candidates'))
ax.set_ylabel(lab('상대 인출 오차', 'relative retrieval error'))
ax.set_title(lab('(d) 근사의 대가 — 정밀 회상의 오염', '(d) recall quality'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_13_7_6')
plt.show()

> ### 읽는 법
>
> (a) 이론 그대로다. 표준 구현의 추가 메모리는 로그–로그 기울기 2, 스트리밍은 1,
> 선형은 길이와 무관한 상수다.
> (b) 이 절의 시험 문제에 대한 답안. 온라인 소프트맥스의 오차는 $10^{-16}$ 수준의
> 반올림 — **같은 함수의 다른 평가 순서**다. 선형 어텐션의 오차는 $10^{0}$ 규모 —
> **다른 함수**다(§13.7.7).
> (c) 정직한 경고 둘. 스트리밍은 중간 길이에서 파이썬 루프 오버헤드로 표준보다
> 느리고, 아주 긴 길이에서야 비슷해진다 — 메모리를 16배 아끼고도 시간은 그대로인
> 것이다. 복잡도만으로 벽시계를 예측할 수 없고(§13.7.8), FlashAttention의 속도는
> 메모리 계층을 겨냥한 구현에서 나온다. 한편 선형 어텐션은 $O(Td^2)$답게 진짜로
> 빠르다(2048에서 20배) — 그 속도의 값이 (b)와 (d)의 오차다.
> (d) 선형 어텐션의 대가가 드러나는 자리. 후보가 늘수록 커널 잔여 가중치가 인출을
> 오염시켜 오차가 자란다 — 임의 쌍 정밀 회상이 필요한 과제에서 희소·선형화가 위험한
> 이유다(§13.7.4, §13.7.7).

---
## 5. 자기 점검

1. (a)의 스트리밍 메모리에서 블록 크기를 8과 512로 바꾸면 곡선이 어떻게 움직이는가? $O(T)$ 항과 $O(\text{block}\cdot T)$ 항을 구분하라.
2. (b)의 스트리밍 오차가 길이에 따라 아주 완만하게 자란다면 그 원인은 무엇인가? (힌트: 부동소수점 덧셈의 결합 순서.)
3. (d)에서 $\beta$(질의의 뾰족함)를 키우면 두 곡선은 각각 어떻게 되는가? 소프트맥스의 지수적 분리와 $\phi$의 다항적 분리를 대조하라.
4. 인과(하삼각) 버전의 선형 어텐션을 누적합으로 구현해 보라(식 13.7.2). 상태 $(S_t,z_t)$의 크기는 얼마인가? §12.7.2의 선형 순환과 어디가 같은가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `block` | 1절 | 64 | 스트리밍의 메모리–속도 교환 |
| `BETA` | 3절 | 6.0 | 회상 과제의 난도 |
| `d` | 1절 | 64 | 커널 특징의 표현력 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")